# Task 1. Data handling and memory management

Thin orchestrator over `mtraffic.io.memory_scenarios` and the partitioned Parquet store.
All heavy logic lives under `src/mtraffic/`; this notebook only calls into it and
renders the headline numbers used in the report.

Run `make ingest` first to populate `data/interim/`.


In [ ]:
from pathlib import Path

import pandas as pd

from mtraffic.config import Config
from mtraffic.io.memory_scenarios import (
    scenario_naive,
    scenario_selective,
    scenario_optimized,
)

cfg = Config.load()
raw_dir = cfg.paths.raw_dir
interim_dir = cfg.paths.interim_dir
report_dir = cfg.paths.reports_dir

## A representative daily file

The three scenarios are measured on the same daily TSV so the comparison is fair.
Each scenario is wrapped by `PeakRSSMonitor` which samples resident set size while
the read runs.


In [ ]:
candidates = sorted(raw_dir.glob('sms-call-internet-mi-*.txt'))
assert candidates, f'No daily TSVs under {raw_dir}'
day_file = candidates[len(candidates) // 2]
print(f'Using {day_file.name} ({day_file.stat().st_size / 2**20:.1f} MB on disk)')

## Run the three scenarios

A. Naive pandas: every column at default dtypes.
B. Selective pandas: 3 columns, explicit dtypes, no aggregation.
C. Optimized streaming: pyarrow batches, column pruning, uint16/float32, group-by-sum.

Note that scenario A allocates roughly 1.3 GB of RSS for a single day. On the 5 GB
raw archive, the naive path would not fit in memory on most laptops.


In [ ]:
results = [scenario_naive(day_file), scenario_selective(day_file), scenario_optimized(day_file)]
df = pd.DataFrame([r.__dict__ for r in results])
df[['scenario', 'rows', 'final_df_mb', 'peak_rss_mb', 'duration_s']].round(2)

## Persisted report

`make ingest` writes the same table to `reports/tables/memory_report.csv` so the
report and the notebook stay in sync.


In [ ]:
csv_path = report_dir / 'tables' / 'memory_report.csv'
if csv_path.exists():
    pd.read_csv(csv_path)
else:
    print(f'{csv_path} not found. Run: make ingest')

## Parquet partition store

Each daily file becomes one Parquet partition under
`data/interim/year_month=YYYY-MM/day=YYYY-MM-DD/part.parquet`. The manifest
records sha256 and row counts.


In [ ]:
manifest = interim_dir / '_manifest.json'
if manifest.exists():
    import json
    m = json.loads(manifest.read_text())
    print(f"Partitions: {len(m.get('partitions', []))}")
    print(f"Total rows: {sum(p['rows'] for p in m.get('partitions', [])):,}")
    print(f"Total bytes: {sum(p['bytes'] for p in m.get('partitions', [])) / 2**20:.1f} MB")
else:
    print(f'{manifest} not found. Run: make ingest')